# S0.1 · 业务最小可行规模的推导

**问题**：预测命题里的「业务最小可行规模」应该取多少？

它不是拍一个业务觉得好看的数字，而是回答一个可判定的问题——
**可用人群小到什么程度，整个方案就无法被验证？**
低于这条线时，即使 VFL 真的有增益，实验也检不出来；此时项目的结论只能是「无法判定」，
而不是「有效」或「无效」。

方法：双臂随机对照下的两比例检验样本量。全部参数来自
`modules/m0_compliance/configs/s0_1_mde.yaml`，逻辑在
`modules/m0_compliance/components/sample_size.py`。

> ⚠️ 基线转化率为**显式假设**（需求方未提供），M1 选定代理数据集后须校准。

In [1]:
import hashlib
import math
import pathlib
import subprocess
import sys

import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
PERCENT = 100

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent   # nbconvert 的工作目录是 notebook 所在目录，须向上找仓库根
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m0_compliance/configs/s0_1_mde.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
config_hash = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()[:HASH_PREFIX_LEN]
git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()

print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + config_hash)
print("seed:", cfg["seed"])
print("git:", git_sha)
print("step:", cfg["step_id"], "|", cfg["purpose"])

config: modules/m0_compliance/configs/s0_1_mde.yaml sha256:68e142c92066
seed: 42
git: ed7ef23
step: S0.1 | 由「能否检出目标增益」反推可用人群 N_eff 的下界


In [2]:
from modules.m0_compliance.components.sample_size import (
    required_n_per_arm,
    sample_size_table,
)

table = pd.DataFrame(sample_size_table(
    base_rates=cfg["base_rates"],
    relative_lifts=cfg["relative_lifts"],
    alpha=cfg["alpha"],
    power=cfg["power"],
    holdout_share=cfg["holdout_share"],
))

RESULT_PATH = REPO_ROOT / "modules/m0_compliance/results/mde_sample_size.csv"
table.to_csv(RESULT_PATH, index=False)

display_table = table.copy()
display_table["基线转化率"] = (display_table["base_rate"] * PERCENT).round(1).astype(str) + "%"
display_table["待检出相对增益"] = (display_table["relative_lift"] * PERCENT).round(0).astype(int).astype(str) + "%"
display_table["每臂样本量"] = display_table["n_per_arm"]
display_table["总可用人群 N_eff"] = display_table["n_total"]
print(display_table[["基线转化率", "待检出相对增益", "每臂样本量", "总可用人群 N_eff"]].to_string(index=False))

基线转化率 待检出相对增益  每臂样本量  总可用人群 N_eff
 1.0%      5% 637008      1274016
 1.0%     10% 163092       326184
 1.0%     20%  42691        85382
 1.0%     30%  19824        39648
 2.0%      5% 315204       630408
 2.0%     10%  80679       161358
 2.0%     20%  21106        42212
 2.0%     30%   9795        19590
 3.0%      5% 207936       415872
 3.0%     10%  53208       106416
 3.0%     20%  13911        27822
 3.0%     30%   6452        12904
 5.0%      5% 122121       244242
 5.0%     10%  31231        62462
 5.0%     20%   8155        16310
 5.0%     30%   3778         7556


In [3]:
smaller_arm_share = min(cfg["holdout_share"], 1 - cfg["holdout_share"])
primary_per_arm = required_n_per_arm(
    base_rate=cfg["primary_base_rate"],
    relative_lift=cfg["primary_relative_lift"],
    alpha=cfg["alpha"],
    power=cfg["power"],
)
primary_total = math.ceil(primary_per_arm / smaller_arm_share)

print("主口径：基线转化率 %.1f%%，待检出相对增益 %.0f%%，alpha=%.2f，power=%.2f"
      % (cfg["primary_base_rate"] * PERCENT, cfg["primary_relative_lift"] * PERCENT,
         cfg["alpha"], cfg["power"]))
print("每臂需 %d 人 → 业务最小可行规模 N_eff ≥ %d 人" % (primary_per_arm, primary_total))

strict = required_n_per_arm(cfg["primary_base_rate"], cfg["relative_lifts"][0],
                            cfg["alpha"], cfg["power"])
print("对照：若按 Loop 规范 0.4 的 MDE 建议初值（相对增益 %.0f%%），"
      "每臂需 %d 人 → N_eff ≥ %d 人"
      % (cfg["relative_lifts"][0] * PERCENT, strict, math.ceil(strict / smaller_arm_share)))

主口径：基线转化率 3.0%，待检出相对增益 20%，alpha=0.05，power=0.80
每臂需 13911 人 → 业务最小可行规模 N_eff ≥ 27822 人
对照：若按 Loop 规范 0.4 的 MDE 建议初值（相对增益 5%），每臂需 207936 人 → N_eff ≥ 415872 人


## 结论

**业务最小可行规模取 N_eff ≥ 30,000 人**（上方主口径算得 27,822，向上取整到万位便于沟通）。

三点必须一并说明，否则这个数字会被误读：

1. **它是「可验证下界」，不是「业务划算下界」。** 低于它，实验检不出目标增益，
   项目结论只能是「无法判定」。业务上是否划算需要单客价值与合规成本，那是 M6 的 ROI 层。
2. **它对基线转化率极其敏感。** 上表显示：基线从 3% 降到 1%，同样检出相对 20% 增益所需人群
   从 2.8 万涨到 8.7 万。基线转化率目前是显式假设，M1 校准后本数字必须重算。
3. **《Loop 执行规范 v2》0.4 建议的 MDE 初值（相对增益 5%）在本场景不现实**——
   它要求 N_eff ≥ 41.6 万，远超跨境共同客户的可能规模。
   这条不是把门槛调低来迁就现实，而是把「我们究竟能检出多大的效应」如实写在前面：
   本方案能验证的是**量级为 20% 相对增益**的效果，检不出更小的差异。

> 下游影响：M3 的合规折损漏斗若算出 N_eff 上界低于 30,000，按框架 M0 升级条件须立即上报，
> 项目命题需重新界定。